### ***LOADING THE FRAMED AUDIO***

In [ ]:
import numpy as np
from pathlib import Path

NOTEBOOK_DIR:Path = Path.cwd()
REPOSITORY_ROOT:Path = NOTEBOOK_DIR
AUDIO_DATA_DIR_FRAMED:Path = REPOSITORY_ROOT / "preprocessing-01-framing.npy"

In [ ]:
LOADED_AUDIO_FRAMES:np.ndarray = np.load(AUDIO_DATA_DIR_FRAMED.resolve(), allow_pickle=True)

In [ ]:
print(LOADED_AUDIO_FRAMES[0])

In [ ]:
n = 25
print("DIR", f"\t{LOADED_AUDIO_FRAMES[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{LOADED_AUDIO_FRAMES[n][1]=}", sep="\n")
print("="*50, "FRAME_DATA", f"\t{LOADED_AUDIO_FRAMES[n][2]=}", sep="\n")

In [ ]:
LOADED_AUDIO_FRAMES[:, 2] = [
    frame * (np.hamming(len(frame)).astype(np.float32))
    for frame in LOADED_AUDIO_FRAMES[:, 2]
]

In [ ]:
n = 25
print("DIR", f"\t{LOADED_AUDIO_FRAMES[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{LOADED_AUDIO_FRAMES[n][1]=}", sep="\n")
print("="*50, "FRAME_DATA", f"\t{LOADED_AUDIO_FRAMES[n][2]=}", sep="\n")

### ***SAVING THE AUDIO FRAME FILES***

In [ ]:
AUDIO_DATA_DIR_WINDOWED:Path = REPOSITORY_ROOT / "preprocessing-02-windowing.npy"
AUDIO_DATA_DIR_WINDOWED.parent.mkdir(parents=True, exist_ok=True)
np.save(AUDIO_DATA_DIR_WINDOWED.resolve(), LOADED_AUDIO_FRAMES, allow_pickle=True)
# np.save normally works only on regular numeric arrays float32 array of shape (n_frames, frame_size).
# if the array has python lists of different lenths
# np has to store it as an object dtype array 
# which needs allow_pickle=True

# note: picking can execute files in the binary so this is a security risk

print(f"Saved {len(LOADED_AUDIO_FRAMES)} frames with shape {LOADED_AUDIO_FRAMES.shape} to {AUDIO_DATA_DIR_WINDOWED.resolve()}")
print("Columns: source_file, frame_number, samples")

### ***VERIFYING THE SAVED WINDOWED FRAMES***

In [ ]:
SAVED_WINDOWED_FRAMES:np.ndarray = np.load(AUDIO_DATA_DIR_WINDOWED.resolve(), allow_pickle=True)

assert SAVED_WINDOWED_FRAMES.shape == LOADED_AUDIO_FRAMES.shape, (
    f"Shape changed: {LOADED_AUDIO_FRAMES.shape} -> {SAVED_WINDOWED_FRAMES.shape}"
)
assert SAVED_WINDOWED_FRAMES.shape == LOADED_AUDIO_FRAMES.shape, (
    "Saved file does not match the frames in memory."
)
assert SAVED_WINDOWED_FRAMES.dtype == LOADED_AUDIO_FRAMES.dtype, (
    f"Data type changed: {LOADED_AUDIO_FRAMES.dtype} -> {SAVED_WINDOWED_FRAMES.dtype}"
)
assert np.array_equal(SAVED_WINDOWED_FRAMES[:, 0], LOADED_AUDIO_FRAMES[:, 0]), (
    "One or more source-file paths changed after windowing."
)
assert np.array_equal(SAVED_WINDOWED_FRAMES[:, 1], LOADED_AUDIO_FRAMES[:, 1]), (
    "One or more frame numbers changed after windowing."
)
for source_file in np.unique(SAVED_WINDOWED_FRAMES[:, 0]):
    frame_numbers = SAVED_WINDOWED_FRAMES[
        SAVED_WINDOWED_FRAMES[:, 0] == source_file, 1
    ].astype(np.int64)
    assert np.array_equal(
        frame_numbers, np.arange(1, len(frame_numbers) + 1)
    ), f"Frame sequence is not continuous for {source_file}"

for saved_row, original_row in zip(SAVED_WINDOWED_FRAMES, LOADED_AUDIO_FRAMES, strict=True):
    saved_frame, original_frame = saved_row[2], original_row[2]

    # frame length must not change, and values must be finite
    assert len(saved_frame) == len(original_frame), (
        f"Wrong frame length in frame {saved_row[1]} of {saved_row[0]}"
    )
    assert np.isfinite(saved_frame).all(), (
        f"NaN or inf in frame {saved_row[1]} of {saved_row[0]}"
    )

print(
    f"Integrity verified: {len(SAVED_WINDOWED_FRAMES)} saved frames"
)